# EGU – InSAR Velocity & Timeseries Viewer

Native-resolution Sentinel-1 InSAR from MintPy (4881×3985 pixels, 80 m, UTM32N).

**Cell 1** – static velocity map  
**Cell 2** – click any pixel → full timeseries pop-up

In [ ]:
%matplotlib widget
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display
from pyproj import Transformer

# ── paths ──────────────────────────────────────────────────────────────────────
BASE = '/mnt/data/aoi_3_bologna/mintpy_filtered/'
VEL_H5  = BASE + 'velocity.h5'
TS_H5   = BASE + 'timeseries_SET_ERA5_ramp_demErr.h5'

# ── grid parameters (UTM 32N) ──────────────────────────────────────────────────
X0, DX =  428520.0,  80.0    # Easting  origin + step (m)
Y0, DY = 5126200.0, -80.0   # Northing origin + step (m)

# ── load velocity + mask ───────────────────────────────────────────────────────
print("Loading velocity …")
with h5py.File(VEL_H5, 'r') as f:
    vel_raw = f['velocity'][:]           # m/yr,  float32

vel_mm = vel_raw * 1000.0               # → mm/yr
vel_mm[vel_mm == 0.0] = np.nan          # zeros are no-data
vel_mm[~np.isfinite(vel_mm)] = np.nan

nrows, ncols = vel_mm.shape
print(f"  grid : {nrows} × {ncols} pixels @ 80 m (UTM 32N)")
print(f"  valid: {np.sum(np.isfinite(vel_mm)):,} pixels")
print(f"  range: {np.nanmin(vel_mm):.1f} to {np.nanmax(vel_mm):.1f} mm/yr")

# ── build UTM coordinate arrays ────────────────────────────────────────────────
col_idx = np.arange(ncols)
row_idx = np.arange(nrows)
easting  = X0 + col_idx * DX               # 1-D
northing = Y0 + row_idx * DY               # 1-D

# ── convert UTM32N → lon/lat (WGS84) ──────────────────────────────────────────
transformer = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)
# sample every 20 pixels for display grid (still 244×199 = 48k points)
STRIDE = 20
ee = easting[::STRIDE]
nn = northing[::STRIDE]
EE, NN = np.meshgrid(ee, nn)
lon2d, lat2d = transformer.transform(EE, NN)
vel_ds = vel_mm[::STRIDE, ::STRIDE]       # downsampled for display

print(f"  display grid (stride={STRIDE}): {vel_ds.shape}")
print(f"  lon range: {lon2d.min():.3f} to {lon2d.max():.3f}")
print(f"  lat range: {lat2d.min():.3f} to {lat2d.max():.3f}")

# ── load acquisition dates ─────────────────────────────────────────────────────
with h5py.File(TS_H5, 'r') as f:
    raw_dates = f['date'][:]
dates = pd.to_datetime([d.decode() for d in raw_dates], format='%Y%m%d')
print(f"  timeseries: {len(dates)} acquisitions  {dates[0].date()} → {dates[-1].date()}")


## 1 · InSAR LOS Velocity Map  (mm/yr)

Blue = subsidence (motion away from sensor) · Red = uplift

In [ ]:
# ── static velocity map ───────────────────────────────────────────────────────
p1, p99 = np.nanpercentile(vel_ds[np.isfinite(vel_ds)], [1, 99])
vlim = max(abs(p1), abs(p99))
norm = mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim)

fig_vel, ax_vel = plt.subplots(figsize=(10, 8), constrained_layout=True)
pcm = ax_vel.pcolormesh(lon2d, lat2d, vel_ds,
                         cmap='RdBu_r', norm=norm, shading='auto',
                         rasterized=True)
cbar = fig_vel.colorbar(pcm, ax=ax_vel, shrink=0.7, pad=0.02)
cbar.set_label('LOS velocity (mm/yr)', fontsize=11)
ax_vel.set_xlabel('Longitude (°E)', fontsize=11)
ax_vel.set_ylabel('Latitude (°N)', fontsize=11)
ax_vel.set_title(
    'Sentinel-1 LOS velocity  |  Bologna / Emilia-Romagna  |  2017–2025\n'
    'MintPy · ERA5+SET+ramp+demErr corrected  ·  ZARVAN-AID GPU ISCE2+',
    fontsize=11
)
plt.show()
print("Velocity map rendered.")


## 2 · Interactive Click-to-Timeseries Viewer

Click any coloured pixel on the map → full LOS displacement timeseries for that pixel is loaded on-demand from the H5 file (no pre-loading the 7 GB array).

In [ ]:
# ── click-to-timeseries interactive viewer ────────────────────────────────────
ts_out = widgets.Output()

fig_click, ax_click = plt.subplots(figsize=(10, 8), constrained_layout=True)
pcm2 = ax_click.pcolormesh(lon2d, lat2d, vel_ds,
                             cmap='RdBu_r', norm=norm, shading='auto',
                             rasterized=True)
cb2 = fig_click.colorbar(pcm2, ax=ax_click, shrink=0.7, pad=0.02)
cb2.set_label('LOS velocity (mm/yr)', fontsize=11)
ax_click.set_xlabel('Longitude (°E)', fontsize=11)
ax_click.set_ylabel('Latitude (°N)', fontsize=11)
ax_click.set_title('Click a pixel to load its full timeseries', fontsize=12)

# scatter to mark clicked point
_click_dot, = ax_click.plot([], [], 'y*', ms=14, zorder=5)

def _lonlat_to_rowcol(lon_click, lat_click):
    """Convert clicked lon/lat → native H5 row/col (nearest neighbour)."""
    # back-project to UTM32N
    e_click, n_click = transformer.transform(lon_click, lat_click,
                                              direction='INVERSE')
    col = int(round((e_click - X0) / DX))
    row = int(round((n_click - Y0) / DY))
    col = max(0, min(col, ncols - 1))
    row = max(0, min(row, nrows - 1))
    return row, col

def _on_click(event):
    if event.inaxes is not ax_click or event.xdata is None:
        return
    lon_c, lat_c = event.xdata, event.ydata
    row, col = _lonlat_to_rowcol(lon_c, lat_c)

    with ts_out:
        ts_out.clear_output(wait=True)
        # on-demand H5 slice – only one pixel's timeseries
        with h5py.File(TS_H5, 'r') as f:
            ts_pixel = f['timeseries'][:, row, col].astype(float) * 1000.0  # m→mm
        vel_px = float(vel_mm[row, col]) if np.isfinite(vel_mm[row, col]) else np.nan
        print(f"  pixel  row={row}, col={col}  |  lon={lon_c:.4f}°, lat={lat_c:.4f}°")
        print(f"  velocity = {vel_px:.1f} mm/yr  |  ts range {np.nanmin(ts_pixel):.1f} to {np.nanmax(ts_pixel):.1f} mm")

        fig_ts, ax_ts = plt.subplots(figsize=(11, 4), constrained_layout=True)
        ax_ts.plot(dates, ts_pixel, lw=1.3, color='#1f77b4', label='LOS (mm)')
        ax_ts.axhline(0, color='k', lw=0.6, linestyle='--', alpha=0.5)
        ax_ts.set_title(
            f'InSAR LOS displacement  ·  ({lat_c:.4f}°N, {lon_c:.4f}°E)  ·  vel={vel_px:.1f} mm/yr',
            fontsize=11
        )
        ax_ts.set_ylabel('Cumulative LOS displacement (mm)', fontsize=10)
        ax_ts.tick_params(axis='x', rotation=30)
        ax_ts.legend(fontsize=9)
        plt.show()

    # update marker on velocity map
    _click_dot.set_data([lon_c], [lat_c])
    fig_click.canvas.draw_idle()

fig_click.canvas.mpl_connect('button_press_event', _on_click)
display(ts_out)
print("Ready – click any pixel on the map above.")
